<a href="https://colab.research.google.com/github/habibiputrar/BigData26_A_2411531001_Habibi-Putra-Rizqullah-/blob/main/Praktikum04/BD_A_TugasP04_2411531001_Habibi_Putra_Rizqullah.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TUGAS PRAKTIKUM BIG DATA 4
## NoSQL dan Pemodelan Data

**Nama:** Habibi Putra Rizqullah  
**NIM:** 2411531001  
**Kelas:** A  


## R. Tugas Praktikum

In [4]:
!pip install -q duckdb networkx tinydb "deltalake>=1.0"

In [5]:
from google.colab import drive
drive.mount("/content/drive")

import os
import json
import time
import shutil
import sqlite3
import zipfile

import pandas as pd
from IPython.display import display, Markdown

DIR_P3 = "/content/drive/MyDrive/BigData/Praktikum3"
DIR_SIMPAN = "/content/drive/MyDrive/BigData/Praktikum4/TugasMandiri"
TRIPS_BERSIH_P4 = f"{DIR_SIMPAN}/trips_bersih"
TRIPS_BERSIH_P3 = f"{DIR_P3}/trips_bersih"
ZIP_TRIPS = f"{DIR_P3}/trips_bersih.zip"

os.makedirs(DIR_SIMPAN, exist_ok=True)

if os.path.isdir(TRIPS_BERSIH_P4):
    TRIPS_BERSIH = TRIPS_BERSIH_P4
else:
    TRIPS_BERSIH = TRIPS_BERSIH_P3

if not os.path.exists(TRIPS_BERSIH) and os.path.exists(ZIP_TRIPS):
    HASIL_EKSTRAK = "/content/trips_bersih_p04"
    os.makedirs(HASIL_EKSTRAK, exist_ok=True)
    with zipfile.ZipFile(ZIP_TRIPS, "r") as arsip:
        arsip.extractall(HASIL_EKSTRAK)
    kandidat = [os.path.join(HASIL_EKSTRAK, "trips_bersih"), HASIL_EKSTRAK]
    TRIPS_BERSIH = next(
        (p for p in kandidat if any(f.endswith(".parquet") for _, _, fs in os.walk(p) for f in fs)),
        TRIPS_BERSIH,
    )

print("Folder Praktikum 4:", DIR_SIMPAN)
print("Dataset:", TRIPS_BERSIH)

catatan = []

def ukur(label, fungsi):
    t0 = time.perf_counter()
    hasil = fungsi()
    detik = time.perf_counter() - t0
    catatan.append({"langkah": label, "detik": round(detik, 4)})
    print(f"[{label}] {detik:.4f} s")
    return hasil

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Folder Praktikum 4: /content/drive/MyDrive/BigData/Praktikum4/TugasMandiri
Dataset: /content/trips_bersih_p04


## R.1 Rekap Benchmark

Tabel rekap K-9 saya isi dari `benchmark_nosql.csv`, hasil dari notebook praktikum. Angka dan kalimat interpretasinya diambil langsung dari berkas itu lewat kode, jadi tidak akan berbeda dari hasil ukur yang tersimpan. Ada beberapa hal yang tidak tercatat di CSV, yaitu jumlah dokumen K-3, hasil k pada graph, hasil Latihan 5, dan hasil Delta Lake. Bagian itu saya ambil dari output notebook praktikum.

Satu hal soal perbandingan yang adil: key-value diukur pada sampel 300.000 baris, sedangkan DuckDB diukur pada seluruh data (2.986.910 baris). Jadi key-value hanya saya bandingkan dengan SQL pada data yang sama, dan DuckDB hanya dengan SQLite pada data yang sama.

In [8]:
PATH_BENCHMARK = "/content/drive/MyDrive/BigData/Praktikum4/benchmark_nosql.csv"
assert os.path.exists(PATH_BENCHMARK), "benchmark_nosql.csv belum ada di folder Praktikum4."

bench = pd.read_csv(PATH_BENCHMARK)
waktu = bench.set_index("langkah")["detik"]
display(bench)

,langkah,detik
0,baca trips_bersih,2.7022
1,relasional: muat tabel,2.5168
2,key-value: bangun dict,9.6995
3,cari 200 perjalanan - SQL tanpa indeks,4.3215
4,cari 200 perjalanan - SQL dengan indeks,0.0037
5,cari 200 perjalanan - key-value,0.0012
6,hitung per wilayah - SQL GROUP BY,0.1951
7,hitung per wilayah - key-value (buka semua nilai),1.4229
8,DuckDB - hitung per wilayah (langsung dari Par...,0.1139
9,SQLite - muat seluruh baris ke tabel,9.1240


In [9]:
def idn(x, desimal=4):
    teks = f"{x:,.{desimal}f}"
    return teks.replace(",", "X").replace(".", ",").replace("X", ".")

def kali(a, b, desimal=1):
    return idn(waktu[a] / waktu[b], desimal)

K_TANPA = "cari 200 perjalanan - SQL tanpa indeks"
K_INDEKS = "cari 200 perjalanan - SQL dengan indeks"
K_KV = "cari 200 perjalanan - key-value"
K_HITUNG_SQL = "hitung per wilayah - SQL GROUP BY"
K_HITUNG_KV = "hitung per wilayah - key-value (buka semua nilai)"
K_DUCK = "DuckDB - hitung per wilayah (langsung dari Parquet)"
K_SQLITE = "SQLite - hitung per wilayah"
K_GRAF = "2 langkah - graph (NetworkX)"
K_SJOIN = "2 langkah - SQL self-join (DuckDB)"
K_LOOP = "cara 1 - loop Python, 5 kotak"
K_NUMPY = "cara 2 - NumPy, 1000 kotak"
K_BANGUN = "bangun indeks STRtree"
K_STR = "cara 3 - STRtree, 1000 kotak"

total_str = waktu[K_BANGUN] + waktu[K_STR]
loop_1000 = waktu[K_LOOP] / 5 * 1000

baris = [
    {
        "Kebutuhan": "Ambil satu data lewat id",
        "Cara yang unggul": "Key-value; tabel dengan indeks",
        "Cara yang lemah": "Tabel tanpa indeks",
        "Angka saya": (
            f"200 pencarian: key-value {idn(waktu[K_KV])} s; tabel berindeks {idn(waktu[K_INDEKS])} s; "
            f"tabel tanpa indeks {idn(waktu[K_TANPA])} s"
        ),
        "Interpretasi": (
            f"Dengan indeks, tabel SQL hanya {kali(K_INDEKS, K_KV)} kali lebih lambat dari key-value, "
            f"dan {kali(K_TANPA, K_INDEKS, 0)} kali lebih cepat daripada yang tanpa indeks. "
            f"Jadi yang menentukan adalah ada tidaknya indeks, bukan jenis modelnya."
        ),
    },
    {
        "Kebutuhan": "Menghitung atau merangkum banyak baris",
        "Cara yang unggul": "Per kolom (DuckDB pada Parquet)",
        "Cara yang lemah": "Key-value (buka semua nilai)",
        "Angka saya": (
            f"300.000 baris: SQL GROUP BY {idn(waktu[K_HITUNG_SQL])} s, key-value {idn(waktu[K_HITUNG_KV])} s. "
            f"2.986.910 baris: DuckDB {idn(waktu[K_DUCK])} s, SQLite {idn(waktu[K_SQLITE])} s"
        ),
        "Interpretasi": (
            f"Key-value {kali(K_HITUNG_KV, K_HITUNG_SQL)} kali lebih lambat dari SQL karena tiap nilai JSON "
            f"harus dibuka dulu. DuckDB {kali(K_SQLITE, K_DUCK)} kali lebih cepat dari SQLite "
            f"karena hanya membaca kolom yang dipakai dan memproses banyak baris sekaligus."
        ),
    },
    {
        "Kebutuhan": "Data dengan field opsional atau berubah-ubah",
        "Cara yang unggul": "Document",
        "Cara yang lemah": "Tabel (kolom NULL, migrasi)",
        "Angka saya": "4.094 dari 20.000 dokumen tanpa field tip; 1 dokumen membawa field cuaca (output K-3)",
        "Interpretasi": (
            "Document menerima field baru atau field yang tidak selalu ada tanpa migrasi. Tapi "
            "salah ketik nama field juga diterima begitu saja tanpa peringatan, jadi validasinya pindah ke aplikasi."
        ),
    },
    {
        "Kebutuhan": "Keterhubungan dengan jumlah langkah bebas",
        "Cara yang unggul": "Graph",
        "Cara yang lemah": "SQL (satu JOIN per langkah)",
        "Angka saya": (
            f"2 langkah: graph {idn(waktu[K_GRAF])} s, SQL {idn(waktu[K_SJOIN])} s. "
            f"Zona terjangkau untuk k = 1 sampai 4: 74, 214, 214, 214 (output K-5)"
        ),
        "Interpretasi": (
            f"Kecepatannya hampir sama (SQL {kali(K_SJOIN, K_GRAF, 2)} kali graph). Kelebihan graph ada di "
            f"cara bertanya: ganti k cukup ubah satu angka, sedangkan SQL harus menambah satu JOIN tiap langkah."
        ),
    },
    {
        "Kebutuhan": "Mencari lokasi dalam wilayah",
        "Cara yang unggul": "Spasial dengan indeks",
        "Cara yang lemah": "Memeriksa semua titik",
        "Angka saya": (
            f"1.000 kotak: STRtree {idn(total_str)} s (termasuk bangun indeks); NumPy {idn(waktu[K_NUMPY])} s; "
            f"loop Python perkiraan {idn(loop_1000, 0)} s"
        ),
        "Interpretasi": (
            f"STRtree {idn(waktu[K_NUMPY] / total_str, 1)} kali lebih cepat dari NumPy dan ribuan kali dari loop. "
            f"Tapi pada kotak 0,02 derajat ia malah 0,68 kali (lebih lambat), dan pada kotak 0,001 derajat 27,5 kali "
            f"(Latihan 5). Jadi klaim percepatan harus menyebut pembandingnya."
        ),
    },
    {
        "Kebutuhan": "Perubahan yang harus aman dan bisa dibatalkan",
        "Cara yang unggul": "Tabel Delta",
        "Cara yang lemah": "Folder Parquet biasa",
        "Angka saya": "Update 20 baris: selisih total_bayar 20,0; kolom asing ditolak, versi tetap 2 (output K-7)",
        "Interpretasi": (
            "Versi lama tetap bisa dibaca setelah update, dan penulisan yang gagal tidak meninggalkan data setengah jadi. "
            "Dua hal ini tidak dimiliki folder Parquet biasa."
        ),
    },
]

rekap = pd.DataFrame(baris)
print("jumlah baris rekap:", len(rekap))

jumlah baris rekap: 6


In [10]:
kolom = ["Kebutuhan", "Cara yang unggul", "Cara yang lemah", "Angka saya", "Interpretasi"]
rekap = rekap[kolom]

tabel_md = ["| " + " | ".join(kolom) + " |", "|" + "---|" * len(kolom)]
for _, r in rekap.iterrows():
    tabel_md.append("| " + " | ".join(str(r[k]).replace("|", "/") for k in kolom) + " |")
display(Markdown("\n".join(tabel_md)))

rekap.to_csv(f"{DIR_SIMPAN}/rekap_k9.csv", index=False)
print("Tersimpan:", f"{DIR_SIMPAN}/rekap_k9.csv")

| Kebutuhan | Cara yang unggul | Cara yang lemah | Angka saya | Interpretasi |
|---|---|---|---|---|
| Ambil satu data lewat id | Key-value; tabel dengan indeks | Tabel tanpa indeks | 200 pencarian: key-value 0,0012 s; tabel berindeks 0,0037 s; tabel tanpa indeks 4,3215 s | Dengan indeks, tabel SQL hanya 3,1 kali lebih lambat dari key-value, dan 1.168 kali lebih cepat daripada yang tanpa indeks. Jadi yang menentukan adalah ada tidaknya indeks, bukan jenis modelnya. |
| Menghitung atau merangkum banyak baris | Per kolom (DuckDB pada Parquet) | Key-value (buka semua nilai) | 300.000 baris: SQL GROUP BY 0,1951 s, key-value 1,4229 s. 2.986.910 baris: DuckDB 0,1139 s, SQLite 1,7269 s | Key-value 7,3 kali lebih lambat dari SQL karena tiap nilai JSON harus dibuka dulu. DuckDB 15,2 kali lebih cepat dari SQLite karena hanya membaca kolom yang dipakai dan memproses banyak baris sekaligus. |
| Data dengan field opsional atau berubah-ubah | Document | Tabel (kolom NULL, migrasi) | 4.094 dari 20.000 dokumen tanpa field tip; 1 dokumen membawa field cuaca (output K-3) | Document menerima field baru atau field yang tidak selalu ada tanpa migrasi. Tapi salah ketik nama field juga diterima begitu saja tanpa peringatan, jadi validasinya pindah ke aplikasi. |
| Keterhubungan dengan jumlah langkah bebas | Graph | SQL (satu JOIN per langkah) | 2 langkah: graph 0,0068 s, SQL 0,0081 s. Zona terjangkau untuk k = 1 sampai 4: 74, 214, 214, 214 (output K-5) | Kecepatannya hampir sama (SQL 1,19 kali graph). Kelebihan graph ada di cara bertanya: ganti k cukup ubah satu angka, sedangkan SQL harus menambah satu JOIN tiap langkah. |
| Mencari lokasi dalam wilayah | Spasial dengan indeks | Memeriksa semua titik | 1.000 kotak: STRtree 0,2927 s (termasuk bangun indeks); NumPy 0,5210 s; loop Python perkiraan 2.794 s | STRtree 1,8 kali lebih cepat dari NumPy dan ribuan kali dari loop. Tapi pada kotak 0,02 derajat ia malah 0,68 kali (lebih lambat), dan pada kotak 0,001 derajat 27,5 kali (Latihan 5). Jadi klaim percepatan harus menyebut pembandingnya. |
| Perubahan yang harus aman dan bisa dibatalkan | Tabel Delta | Folder Parquet biasa | Update 20 baris: selisih total_bayar 20,0; kolom asing ditolak, versi tetap 2 (output K-7) | Versi lama tetap bisa dibaca setelah update, dan penulisan yang gagal tidak meninggalkan data setengah jadi. Dua hal ini tidak dimiliki folder Parquet biasa. |

Tersimpan: /content/drive/MyDrive/BigData/Praktikum4/TugasMandiri/rekap_k9.csv


## R.2 Data Lain, Cara Lain: Data Zona

Data yang saya pilih adalah data zona, yaitu nomor zona, wilayahnya, dan arus perjalanan ke zona lain. Data ini saya simpan dengan tiga cara: tabel (SQLite), document (TinyDB), dan graph (NetworkX). Tiap cara saya coba dengan dua pertanyaan, satu yang gampang dijawab dan satu yang canggung. Hasilnya saya cocokkan antar cara untuk memastikan ketiganya menjawab hal yang sama.

In [11]:
import glob
import duckdb
import networkx as nx
from tinydb import TinyDB, Query
from tinydb.storages import MemoryStorage

GLOB = f"{TRIPS_BERSIH}/*/*.parquet"
if not glob.glob(GLOB):
    GLOB = f"{TRIPS_BERSIH}/**/*.parquet"

con_d = duckdb.connect()
con_d.sql(f"""
    CREATE OR REPLACE VIEW perjalanan AS
    SELECT tpep_pickup_datetime AS waktu_jemput,
           PULocationID         AS zona_jemput,
           DOLocationID         AS zona_tujuan,
           CASE WHEN borough_naik IS NULL
                OR borough_naik = '__HIVE_DEFAULT_PARTITION__'
                THEN 'Unknown' ELSE borough_naik END AS wilayah_jemput,
           trip_distance        AS jarak_mil,
           total_amount         AS total_bayar,
           nama_pembayaran      AS metode_bayar
    FROM read_parquet('{GLOB}', hive_partitioning=true)
""")

arus = ukur("hitung arus antar zona", lambda: con_d.sql("""
    SELECT zona_jemput AS asal, zona_tujuan AS tujuan, COUNT(*) AS jumlah
    FROM perjalanan
    GROUP BY 1, 2
""").df())

zona_ref = con_d.sql("""
    SELECT zona_jemput AS zona, MIN(wilayah_jemput) AS wilayah
    FROM perjalanan
    GROUP BY 1
""").df()

arus["asal"] = arus["asal"].astype(int)
arus["tujuan"] = arus["tujuan"].astype(int)
arus["jumlah"] = arus["jumlah"].astype(int)
zona_ref["zona"] = zona_ref["zona"].astype(int)
wilayah_zona = dict(zip(zona_ref["zona"], zona_ref["wilayah"]))

populer = (
    arus.groupby("tujuan", as_index=False)["jumlah"].sum()
    .sort_values(["jumlah", "tujuan"], ascending=[False, True])
)
zona_awal = int(populer.iloc[0]["tujuan"])
MIN = 50

print("zona:", len(zona_ref), "| arus:", len(arus), "| zona awal (tujuan terpopuler):", zona_awal)

[hitung arus antar zona] 0.2041 s
zona: 254 | arus: 21583 | zona awal (tujuan terpopuler): 236


### R.2.1 Tiga cara menyimpan

Untuk tabel saya pakai dua tabel SQLite, `zona(zona, wilayah)` dan `arus(asal, tujuan, jumlah)`. Untuk document, satu dokumen mewakili satu zona dan isinya wilayah beserta seluruh daftar tujuannya (embedding). Untuk graph, zona jadi node, arus jadi edge berarah dengan bobot `jumlah`, dan wilayah disimpan sebagai atribut node.

In [12]:
# 1. Tabel
con_s = sqlite3.connect(":memory:")
zona_ref.to_sql("zona", con_s, index=False)
arus.to_sql("arus", con_s, index=False)
con_s.execute("CREATE INDEX idx_arus_asal ON arus(asal)")

# 2. Document
db = TinyDB(storage=MemoryStorage)
tabel_zona = db.table("zona")
keluar = {}
urut = arus.sort_values(["asal", "jumlah", "tujuan"], ascending=[True, False, True])
for r in urut.itertuples(index=False):
    keluar.setdefault(int(r.asal), []).append({"tujuan": int(r.tujuan), "jumlah": int(r.jumlah)})

tabel_zona.insert_multiple([
    {
        "zona": z,
        "wilayah": wilayah_zona.get(z, "Unknown"),
        "total_keluar": sum(t["jumlah"] for t in daftar),
        "keluar": daftar,
    }
    for z, daftar in keluar.items()
])

# 3. Graph
G = nx.DiGraph()
for r in arus.itertuples(index=False):
    G.add_edge(int(r.asal), int(r.tujuan), jumlah=int(r.jumlah))
nx.set_node_attributes(G, wilayah_zona, "wilayah")

print("tabel zona:", con_s.execute("SELECT COUNT(*) FROM zona").fetchone()[0],
      "| tabel arus:", con_s.execute("SELECT COUNT(*) FROM arus").fetchone()[0])
print("dokumen zona:", len(tabel_zona))
print("node:", G.number_of_nodes(), "| edge:", G.number_of_edges())

tabel zona: 254 | tabel arus: 21583
dokumen zona: 254
node: 262 | edge: 21583


### R.2.2 Tabel: satu pertanyaan mudah, satu canggung

Yang mudah: berapa total perjalanan yang keluar dari tiap wilayah. Di SQL ini cukup satu JOIN dan satu GROUP BY.

In [13]:
def keluar_per_wilayah_tabel():
    return dict(con_s.execute("""
        SELECT z.wilayah, SUM(a.jumlah)
        FROM arus a
        JOIN zona z ON z.zona = a.asal
        GROUP BY z.wilayah
    """).fetchall())

hasil_tabel_mudah = ukur("tabel (mudah): perjalanan keluar per wilayah", keluar_per_wilayah_tabel)
print(sorted(hasil_tabel_mudah.items(), key=lambda x: (-x[1], x[0])))

[tabel (mudah): perjalanan keluar per wilayah] 0.0258 s
[('Manhattan', 2659609), ('Queens', 270315), ('Unknown', 37949), ('Brooklyn', 15724), ('Bronx', 3074), ('Staten Island', 211), ('EWR', 28)]


Yang canggung: zona apa saja yang bisa dicapai dalam paling banyak 3 langkah dari zona awal. Setiap tambahan langkah berarti satu self-join lagi, dan hasil dari tiap panjang jalur harus digabung dengan UNION.

In [14]:
def tiga_langkah_tabel():
    baris = con_s.execute("""
        WITH a AS (SELECT asal, tujuan FROM arus WHERE jumlah >= :m)
        SELECT tujuan FROM a WHERE asal = :s
        UNION
        SELECT a2.tujuan
        FROM a a1 JOIN a a2 ON a1.tujuan = a2.asal
        WHERE a1.asal = :s
        UNION
        SELECT a3.tujuan
        FROM a a1 JOIN a a2 ON a1.tujuan = a2.asal
                  JOIN a a3 ON a2.tujuan = a3.asal
        WHERE a1.asal = :s
    """, {"s": zona_awal, "m": MIN}).fetchall()
    return {r[0] for r in baris} - {zona_awal}

hasil_tabel_canggung = ukur("tabel (canggung): terjangkau <= 3 langkah", tiga_langkah_tabel)
print("zona terjangkau:", len(hasil_tabel_canggung))

[tabel (canggung): terjangkau <= 3 langkah] 0.2718 s
zona terjangkau: 214


### R.2.3 Document: satu pertanyaan mudah, satu canggung

Yang mudah: menampilkan profil lengkap satu zona (wilayah dan semua tujuannya). Semuanya sudah ada dalam satu dokumen, jadi cukup satu kali baca.

In [15]:
Q = Query()

def profil_zona(z):
    return tabel_zona.get(Q.zona == z)

dok = ukur("document (mudah): profil satu zona", lambda: profil_zona(zona_awal))
print("zona:", dok["zona"], "| wilayah:", dok["wilayah"],
      "| tujuan berbeda:", len(dok["keluar"]), "| total keluar:", dok["total_keluar"])
print("tiga tujuan teratas:", dok["keluar"][:3])

[document (mudah): profil satu zona] 0.0002 s
zona: 236 | wilayah: Manhattan | tujuan berbeda: 203 | total keluar: 136276
tiga tujuan teratas: [{'tujuan': 237, 'jumlah': 18810}, {'tujuan': 236, 'jumlah': 14110}, {'tujuan': 161, 'jumlah': 7497}]


Yang canggung: zona mana yang paling banyak mengirim perjalanan ke zona awal, jadi dari arah masuk. Dokumennya disusun per zona asal, sehingga untuk arah sebaliknya semua dokumen harus dibuka dan daftar tujuannya dicek satu per satu.

In [16]:
def lima_pengirim_terbesar_doc(z):
    hasil = []
    for d in tabel_zona.all():
        for t in d["keluar"]:
            if t["tujuan"] == z:
                hasil.append((d["zona"], t["jumlah"]))
    return sorted(hasil, key=lambda x: (-x[1], x[0]))[:5]

def lima_pengirim_terbesar_sql(z):
    return con_s.execute("""
        SELECT asal, jumlah FROM arus
        WHERE tujuan = ?
        ORDER BY jumlah DESC, asal
        LIMIT 5
    """, (z,)).fetchall()

h_doc = ukur("document (canggung): pengirim terbesar ke satu zona", lambda: lima_pengirim_terbesar_doc(zona_awal))
h_sql = lima_pengirim_terbesar_sql(zona_awal)
print("document:", h_doc)
print("SQL     :", [tuple(r) for r in h_sql])
print("hasil sama?", h_doc == [tuple(r) for r in h_sql])

[document (canggung): pengirim terbesar ke satu zona] 0.0051 s
document: [(237, 22065), (236, 14110), (161, 8524), (141, 8232), (263, 6371)]
SQL     : [(237, 22065), (236, 14110), (161, 8524), (141, 8232), (263, 6371)]
hasil sama? True


### R.2.4 Graph: satu pertanyaan mudah, satu canggung

Yang mudah: zona apa saja yang bisa dicapai dalam paling banyak 3 langkah. Cukup menulis `cutoff=3`, lalu hasilnya saya cocokkan dengan jawaban tabel di R.2.2.

In [17]:
H = nx.DiGraph([(a, b) for a, b, d in G.edges(data=True) if d["jumlah"] >= MIN])

def tiga_langkah_graph():
    if zona_awal not in H:
        return set()
    return set(nx.single_source_shortest_path_length(H, zona_awal, cutoff=3)) - {zona_awal}

hasil_graph_mudah = ukur("graph (mudah): terjangkau <= 3 langkah", tiga_langkah_graph)
print("zona terjangkau:", len(hasil_graph_mudah))
print("sama dengan jawaban tabel?", hasil_graph_mudah == hasil_tabel_canggung)

[graph (mudah): terjangkau <= 3 langkah] 0.0005 s
zona terjangkau: 214
sama dengan jawaban tabel? True


Yang canggung: total perjalanan yang keluar dari tiap wilayah. Graph tidak punya GROUP BY, jadi semua edge harus dilewati satu per satu lalu dijumlahkan sendiri berdasarkan atribut wilayah node asalnya. Pertanyaan seperti rata-rata `total_bayar` per wilayah malah tidak bisa dijawab dari graph ini, karena nilainya tidak disimpan di node maupun edge.

In [18]:
def keluar_per_wilayah_graph():
    total = {}
    for asal, tujuan, d in G.edges(data=True):
        w = G.nodes[asal]["wilayah"]
        total[w] = total.get(w, 0) + d["jumlah"]
    return total

hasil_graph_canggung = ukur("graph (canggung): perjalanan keluar per wilayah", keluar_per_wilayah_graph)
print(sorted(hasil_graph_canggung.items(), key=lambda x: (-x[1], x[0])))
print("sama dengan jawaban tabel?", hasil_graph_canggung == hasil_tabel_mudah)

[graph (canggung): perjalanan keluar per wilayah] 0.0170 s
[('Manhattan', 2659609), ('Queens', 270315), ('Unknown', 37949), ('Brooklyn', 15724), ('Bronx', 3074), ('Staten Island', 211), ('EWR', 28)]
sama dengan jawaban tabel? True


### R.2.5 Ringkasan

| Cara menyimpan | Pertanyaan yang mudah | Pertanyaan yang canggung | Alasan |
|---|---|---|---|
| Tabel | Total perjalanan keluar per wilayah | Zona terjangkau dalam paling banyak 3 langkah | Agregasi memang kerjaan SQL (JOIN dan GROUP BY). Jalur berlapis butuh satu self-join per langkah plus UNION. |
| Document | Profil lengkap satu zona | Pengirim terbesar ke satu zona (arah masuk) | Dokumen disusun per zona asal. Satu zona bisa dibaca sekali jalan, tapi untuk arah sebaliknya semua dokumen harus diperiksa. |
| Graph | Zona terjangkau dalam paling banyak k langkah | Total perjalanan per wilayah | Mencari tetangga cukup dengan mengganti k. Agregasi per kelompok harus ditulis sendiri karena graph tidak punya GROUP BY. |

Dengan hanya 262 zona, selisih waktu ketiga cara sangat kecil, jadi tidak bisa dipakai untuk memilih. Yang membedakan adalah bentuk pertanyaannya: tiap cara murah untuk satu jenis pertanyaan dan mahal untuk jenis lain. Hasil yang sudah dicocokkan menunjukkan ketiganya memberi jawaban yang sama, jadi pilihannya tergantung pertanyaan mana yang paling sering diajukan.

## R.3 Delta Lake Dua Hari

Dua hari kerja saya simulasikan sebagai dua tahap pada tabel Delta yang sama, memakai 100.000 baris sampel dari `trips_bersih`. Hari pertama ada dua penulisan, yaitu batch pagi 40.000 baris (versi 0) dan batch sore 20.000 baris (versi 1), lalu keadaan akhirnya dicatat. Hari kedua menambah 40.000 baris (versi 2), kemudian ada koreksi tarif Tunai di wilayah Queens sebesar 1,0 (versi 3). Tiap komit saya beri label hari lewat `custom_metadata` supaya kelihatan di riwayat.

In [19]:
from deltalake import DeltaTable, write_deltalake, CommitProperties

data_delta = con_d.sql("""
    SELECT waktu_jemput, zona_jemput, zona_tujuan, wilayah_jemput,
           jarak_mil, total_bayar, metode_bayar
    FROM perjalanan
    USING SAMPLE reservoir(100000 ROWS) REPEATABLE (42)
""").df().reset_index(drop=True)
print("baris sampel:", f"{len(data_delta):,}")

PATH_2HARI = os.path.join(os.getcwd(), "trips_delta_2hari")
if os.path.exists(PATH_2HARI):
    shutil.rmtree(PATH_2HARI)

def label(hari, ket):
    return CommitProperties(custom_metadata={"hari": str(hari), "keterangan": ket})

baris sampel: 100,000


### R.3.1 Hari pertama

In [20]:
pagi = data_delta.iloc[:40_000]
sore = data_delta.iloc[40_000:60_000]

write_deltalake(PATH_2HARI, pagi, mode="overwrite", commit_properties=label(1, "batch pagi"))
write_deltalake(PATH_2HARI, sore, mode="append", commit_properties=label(1, "batch sore"))

dt = DeltaTable(PATH_2HARI)
VERSI_AKHIR_HARI1 = dt.version()
akhir_hari1 = dt.to_pandas()
BARIS_HARI1 = len(akhir_hari1)
TOTAL_HARI1 = float(akhir_hari1["total_bayar"].sum())

print("versi akhir hari 1:", VERSI_AKHIR_HARI1)
print("baris akhir hari 1:", f"{BARIS_HARI1:,}")
print("total_bayar akhir hari 1:", round(TOTAL_HARI1, 2))

versi akhir hari 1: 1
baris akhir hari 1: 60,000
total_bayar akhir hari 1: 1559049.94


### R.3.2 Hari kedua

In [21]:
hari2 = data_delta.iloc[60_000:100_000]
write_deltalake(PATH_2HARI, hari2, mode="append", commit_properties=label(2, "tambah data hari kedua"))

tersedia = data_delta.iloc[:100_000]
n_koreksi = int(((tersedia["wilayah_jemput"] == "Queens") & (tersedia["metode_bayar"] == "Tunai")).sum())
assert n_koreksi > 0, "Tidak ada baris Queens + Tunai pada sampel. Ganti predikat koreksi."
print("baris yang akan dikoreksi:", n_koreksi)

dt = DeltaTable(PATH_2HARI)
dt.update(
    predicate="wilayah_jemput = 'Queens' AND metode_bayar = 'Tunai'",
    updates={"total_bayar": "total_bayar + 1.0"},
    commit_properties=label(2, "koreksi tarif Tunai Queens"),
)

terbaru = DeltaTable(PATH_2HARI).to_pandas()
print("versi terbaru:", DeltaTable(PATH_2HARI).version(), "| baris:", f"{len(terbaru):,}")

baris yang akan dikoreksi: 704
versi terbaru: 3 | baris: 100,000


### R.3.3 Membaca tabel seperti keadaan akhir hari pertama (time travel)

In [22]:
h1 = DeltaTable(PATH_2HARI, version=VERSI_AKHIR_HARI1).to_pandas()
total_h1_baca = float(h1["total_bayar"].sum())

print("time travel ke versi", VERSI_AKHIR_HARI1)
print("baris hasil baca      :", f"{len(h1):,}", "| catatan akhir hari 1:", f"{BARIS_HARI1:,}")
print("total_bayar hasil baca:", round(total_h1_baca, 2), "| catatan akhir hari 1:", round(TOTAL_HARI1, 2))
print("sama dengan keadaan akhir hari 1?",
      len(h1) == BARIS_HARI1 and abs(total_h1_baca - TOTAL_HARI1) < 0.01)

selisih = float(terbaru["total_bayar"].sum()) - total_h1_baca
harapan = float(hari2["total_bayar"].sum()) + n_koreksi * 1.0
print("selisih total_bayar (terbaru - akhir hari 1):", round(selisih, 2))
print("total hari 2 + koreksi                      :", round(harapan, 2))
print("selisih sesuai?", abs(selisih - harapan) < 0.01)

time travel ke versi 1
baris hasil baca      : 60,000 | catatan akhir hari 1: 60,000
total_bayar hasil baca: 1559049.94 | catatan akhir hari 1: 1559049.94
sama dengan keadaan akhir hari 1? True
selisih total_bayar (terbaru - akhir hari 1): 1038019.97
total hari 2 + koreksi                      : 1038019.97
selisih sesuai? True


### R.3.4 Riwayat dan isi `_delta_log/`

In [23]:
for h in DeltaTable(PATH_2HARI).history():
    print(h.get("version"), h.get("operation"), "| hari:", h.get("hari"), "|", h.get("keterangan"))

3 UPDATE | hari: 2 | koreksi tarif Tunai Queens
2 WRITE | hari: 2 | tambah data hari kedua
1 WRITE | hari: 1 | batch sore
0 WRITE | hari: 1 | batch pagi


In [24]:
PATH_LOG = f"{PATH_2HARI}/_delta_log"
berkas_log = sorted(f for f in os.listdir(PATH_LOG) if f.endswith(".json"))
print("berkas log:", berkas_log)

for nama in berkas_log:
    print("\n" + "=" * 70)
    print(nama)
    print("=" * 70)
    with open(f"{PATH_LOG}/{nama}", encoding="utf-8") as f:
        for baris in f:
            print(baris.rstrip())

berkas log: ['00000000000000000000.json', '00000000000000000001.json', '00000000000000000002.json', '00000000000000000003.json']

00000000000000000000.json
{"commitInfo":{"timestamp":1791200262737,"operation":"WRITE","operationParameters":{"mode":"Overwrite"},"engineInfo":"delta-rs:py-1.6.6","keterangan":"batch pagi","hari":"1","operationMetrics":{"num_added_files":1,"num_removed_files":0,"num_partitions":0,"num_added_rows":40000,"execution_time_ms":21,"num_retries":0},"clientVersion":"delta-rs.py-1.6.6"}}
{"protocol":{"minReaderVersion":3,"minWriterVersion":7,"readerFeatures":["timestampNtz"],"writerFeatures":["timestampNtz"]}}
{"metaData":{"id":"0fc0f4fd-9df1-49cb-b789-05947cea4f88","name":null,"description":null,"format":{"provider":"parquet","options":{}},"schemaString":"{\"type\":\"struct\",\"fields\":[{\"name\":\"waktu_jemput\",\"type\":\"timestamp_ntz\",\"nullable\":true,\"metadata\":{}},{\"name\":\"zona_jemput\",\"type\":\"long\",\"nullable\":true,\"metadata\":{}},{\"name\":\"z

In [25]:
ringkas = []
for nama in berkas_log:
    ditambah, dihapus, operasi = 0, 0, None
    with open(f"{PATH_LOG}/{nama}", encoding="utf-8") as f:
        for baris in f:
            aksi = json.loads(baris)
            if "add" in aksi:
                ditambah += 1
            elif "remove" in aksi:
                dihapus += 1
            elif "commitInfo" in aksi:
                operasi = aksi["commitInfo"].get("operation")
    ringkas.append({"berkas": nama, "operasi": operasi, "file ditambah": ditambah, "file dihapus": dihapus})

display(pd.DataFrame(ringkas))

,berkas,operasi,file ditambah,file dihapus
0,00000000000000000000.json,WRITE,1,0
1,00000000000000000001.json,WRITE,1,0
2,00000000000000000002.json,WRITE,1,0
3,00000000000000000003.json,UPDATE,1,3


### R.3.5 Arsip untuk dikumpulkan

In [26]:
PATH_ZIP_2HARI = f"{DIR_SIMPAN}/trips_delta_2hari"
hasil_zip = shutil.make_archive(PATH_ZIP_2HARI, "zip", root_dir=PATH_2HARI)
print("Arsip tersimpan:", hasil_zip)

Arsip tersimpan: /content/drive/MyDrive/BigData/Praktikum4/TugasMandiri/trips_delta_2hari.zip


## S. Pertanyaan Evaluasi

### S.1 Mengapa NoSQL sering dibaca "Not Only SQL"?

NoSQL bukan pengganti SQL. Ia menambah pilihan cara menyimpan data, jadi satu sistem bisa memakai beberapa jenis penyimpanan sekaligus, masing-masing untuk pekerjaan yang paling cocok (polyglot persistence). Di praktikum ini kelihatan jelas: key-value untuk cari satu id, document untuk data yang bentuknya berbeda-beda, DuckDB untuk merangkum banyak baris, graph untuk keterhubungan, sementara SQL tetap dipakai sebagai pembanding di hampir semua percobaan.

Dua situasi ketika tabel relasional tetap lebih baik:

1. **Transaksi dan hubungan antar data yang harus selalu konsisten.** Data pembayaran atau tarif perlu jaminan ACID dan sering digabung lewat JOIN. Tabel relasional memang dibuat untuk itu, sedangkan document store lemah untuk transaksi lintas dokumen.
2. **Pertanyaan yang belum bisa diduga di awal.** Pada SQL kita bebas menulis query baru kapan saja. Di K-2, tabel SQL yang diberi indeks hanya sekitar 3,1 kali lebih lambat dari key-value untuk cari id (0,0037 detik lawan 0,0012 detik), dan untuk menghitung rata-rata per wilayah SQL justru sekitar 7,3 kali lebih cepat dari key-value (0,1951 detik lawan 1,4229 detik). Kalau pertanyaannya beragam, SQL lebih aman dipilih.



### S.2 Perbedaan BASE dan ACID, serta teorema CAP

ACID (Atomicity, Consistency, Isolation, Durability) menjamin satu transaksi berhasil seluruhnya atau tidak sama sekali, dan data selalu dalam keadaan valid. BASE (Basically Available, Soft state, Eventually consistent) mengutamakan ketersediaan dan kecepatan. Data boleh sesaat berbeda antar salinan, asalkan akhirnya seragam.

Teorema CAP bicara soal saat jaringan antar mesin terputus (partition). Pada kondisi itu sistem harus memilih: consistency, artinya semua pembaca melihat data terbaru yang sama, atau availability, artinya setiap permintaan tetap dijawab walaupun datanya mungkin belum terbaru.

Contoh di aplikasi taksi:

- **Memilih consistency:** status pembayaran sebuah perjalanan dan koreksi tarif untuk audit keuangan. Lebih baik permintaan ditunda atau ditolak sebentar daripada tim keuangan atau penumpang melihat angka yang salah. Di praktikum ini Delta Lake mewakili sifat ini: penambahan kolom asing ditolak dan versi tabel tetap 2, tidak ada data setengah jadi.
- **Boleh memilih availability:** daftar "perjalanan terbaru" atau rekomendasi zona yang sering dituju. Kalau data tertinggal beberapa detik, pengguna hampir tidak merasakannya, dan halaman yang tetap terbuka lebih penting.

### S.3 Mengapa key-value cepat untuk cari satu data tetapi lambat untuk menghitung?

Pada key-value, kunci langsung menunjuk letak datanya, seperti nomor loker. Tidak ada baris yang diperiksa satu per satu, jadi 200 pencarian id selesai dalam 0,0012 detik di K-2.

Untuk menghitung, nilai di key-value hanyalah sepotong teks JSON bagi penyimpannya. Database tidak tahu mana yang kolom `total_bayar`, sehingga kode harus membuka 300.000 nilai dengan `json.loads` satu per satu, lalu menjumlahkannya sendiri. Hasilnya 1,4229 detik, sekitar 7,3 kali lebih lambat dari SQL `GROUP BY` (0,1951 detik) yang sudah tahu bahwa `total_bayar` adalah kolom angka.

Yang dikorbankan key-value demi kesederhanaannya adalah kemampuan bertanya berdasarkan isi data: filter, agregasi, dan JOIN. Semuanya harus ditulis sendiri oleh aplikasi. Perlu diingat juga bahwa di satu komputer keunggulan key-value bukan terutama kecepatan (tabel berindeks hampir sama cepat), tetapi kemudahan membagi data ke banyak mesin karena kuncinya sudah menentukan lokasinya.

### S.4 Menyatukan (embedding) dan merujuk (referencing) pada model document

**Embedding** berarti data terkait dimasukkan ke dalam satu dokumen. Pada K-3, zona dan wilayah jemput ikut tersimpan di dokumen perjalanan, sehingga sekali baca semuanya sudah ada. Harganya data jadi terduplikasi: kalau nama sebuah wilayah berubah, semua dokumen yang memuatnya harus diperbarui.

**Referencing** berarti dokumen hanya menyimpan id (misalnya nomor zona) dan detailnya disimpan di tempat lain. Hemat dan mudah diperbarui, tetapi menampilkan wilayah butuh pencarian kedua.

Embedding lebih tepat untuk data yang dibaca bersama dan jarang berubah, misalnya halaman detail perjalanan yang menampilkan zona jemput dan biaya sekaligus. Referencing lebih tepat untuk data yang sering berubah atau dipakai banyak dokumen, misalnya keterangan zona yang dirujuk oleh jutaan perjalanan.


### S.5 Kapan graph lebih unggul dari SQL untuk keterhubungan?

Graph unggul ketika pertanyaannya soal jalur dengan jumlah langkah yang bebas atau berubah-ubah, misalnya "zona mana saja yang bisa dicapai dalam paling banyak k langkah". Alasannya bukan kecepatan, karena di K-5 graph (0,0068 detik) dan SQL (0,0081 detik) sama-sama cepat pada 262 zona.

Alasannya ada pada cara pertanyaan ditulis. Di SQL, setiap tambahan langkah butuh satu self-join lagi, dan di R.2 tiga langkah sudah memaksa kami menulis tiga bentuk query yang digabung dengan `UNION`. Di graph cukup mengganti `cutoff=k`, dan hasilnya langsung terbaca untuk k = 1 sampai 4 (74, 214, 214, 214 zona). Bentuk data (titik dan garis) juga sama dengan bentuk masalahnya, sehingga query lebih mudah dibaca dan dirawat. Kebalikannya, graph canggung untuk agregasi per kelompok seperti total perjalanan per wilayah, karena tidak punya `GROUP BY`.


### S.6 Indeks spasial dan klaim "1.000 kali lebih cepat"

Sebelum memeriksa lokasi secara tepat, indeks spasial (STRtree, sejenis R-tree) mengelompokkan titik yang berdekatan ke dalam kotak berjenjang. Saat mencari, kelompok yang kotaknya tidak bersinggungan dengan wilayah pencarian dilewati begitu saja. Pemeriksaan tepat (`contains`) hanya dilakukan pada titik-titik kandidat yang tersisa. Cara kerjanya mirip partition pruning di Praktikum 3, hanya saja pemisahannya berdasarkan letak di peta.

Klaim "indeks 1.000 kali lebih cepat" menyesatkan karena angkanya bergantung pada pembanding dan kondisi pengujian. Dari hasil saya:

- terhadap loop Python, STRtree sekitar 9.500 kali lebih cepat (0,2927 detik lawan perkiraan 2.794 detik untuk 1.000 kotak),
- terhadap NumPy, hanya sekitar 1,8 kali (0,5210 detik),
- pada kotak 0,02 derajat STRtree malah lebih lambat dari NumPy (rasio 0,68), sedangkan pada kotak 0,001 derajat sekitar 27 kali lebih cepat.

Jadi klaim percepatan harus menyebut pembandingnya, ukuran kotak, jumlah titik, dan apakah waktu membangun indeks (0,0788 detik) ikut dihitung. Titik yang dipakai juga sintetis, jadi angkanya menggambarkan cara kerja indeks, bukan pola perjalanan taksi yang sebenarnya.

### S.7 Isi `_delta_log/`, atomik, time travel, dan bahaya VACUUM retention 0 jam

Folder `_delta_log/` berisi berkas JSON bernomor, satu berkas untuk tiap komit (`00000000000000000000.json`, `...01.json`, dan seterusnya). Isinya `commitInfo` (jenis operasi dan metrik), skema tabel, serta daftar aksi `add` untuk file Parquet yang masuk ke tabel dan `remove` untuk file yang dikeluarkan. Di R.3, tabel dua hari punya empat berkas log. Versi 3 (UPDATE) tercatat menambah 1 file dan mengeluarkan 3 file.

**Atomik.** Satu komit adalah satu berkas yang ditulis sekaligus, dan file Parquet baru baru dianggap bagian tabel setelah dicatat di log. Pembaca jadi melihat tabel sebelum atau sesudah komit, tidak pernah di tengahnya. Kalau penulisan gagal, tidak ada berkas log baru. Buktinya di K-7: append dengan kolom asing ditolak dan versi tabel tetap 2.

**Time travel.** Isi tabel pada versi n adalah hasil membaca log sampai berkas ke-n. Update dan compaction tidak mengubah file lama, hanya menulis file baru lalu mencatat file lama sebagai `remove`, jadi versi sebelumnya masih bisa direkonstruksi. Di R.3, membaca versi 1 memberi 60.000 baris dengan total `total_bayar` yang sama persis dengan catatan akhir hari pertama.

**VACUUM retention 0 jam** berbahaya karena langsung menghapus secara fisik semua file yang sudah tidak aktif, tanpa masa tunggu. File yang mungkin masih dibaca proses lain ikut hilang, dan time travel atau restore ke versi lama terputus karena file yang dibutuhkannya sudah tidak ada. Penghapusan itu tidak bisa dibatalkan. Di K-8 saya hanya memakai `dry_run=True`, yang melaporkan 23 file akan dihapus tanpa menghapus apa pun. Nilai bawaan sekitar 7 hari ada untuk melindungi hal ini.